In [1]:
from datasets import load_dataset
import soundfile as sf
import os
import io


/Users/dvir.benor/PycharmProjects/audio-classification-playground/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import sys
sys.executable

'/Users/dvir.benor/PycharmProjects/audio-classification-playground/.venv/bin/python'

In [4]:
target_dir = "../resources/speech"

# Create a folder for your samples

os.makedirs(target_dir, exist_ok=True)

# 1. Load in streaming mode (no full download)
# We'll use 'train.360' to ensure we find longer clips quickly
ds = load_dataset("librispeech_asr", "clean", split="train.360", streaming=True)

found_count = 0
target_count = 5
min_duration_sec = 15

print(f"Searching for samples >= {min_duration_sec}s...")

for sample in ds:
    audio = sample["audio"]
    array = audio["array"]
    sampling_rate = audio["sampling_rate"]
    
    # Calculate duration: total samples / samples per second
    duration = len(array) / sampling_rate
    
    if duration >= min_duration_sec:
        found_count += 1
        file_name = os.path.join(target_dir,f"sample_{found_count}.wav")
        
        # Save as WAV
        sf.write(file_name, array, sampling_rate)
        
        print(f"Saved {file_name} (Duration: {duration:.2f}s)")
    else:
        print(f"Skipping - (Duration: {duration:.2f}s)")
        
    if found_count >= target_count:
        break

print("Done!")

Searching for samples >= 15s...
Skipping - (Duration: 14.12s)
Skipping - (Duration: 12.71s)
Skipping - (Duration: 13.95s)
Skipping - (Duration: 13.30s)
Saved ../resources/speech/sample_1.wav (Duration: 16.77s)
Saved ../resources/speech/sample_2.wav (Duration: 15.18s)
Saved ../resources/speech/sample_3.wav (Duration: 16.05s)
Skipping - (Duration: 14.84s)
Saved ../resources/speech/sample_4.wav (Duration: 15.42s)
Saved ../resources/speech/sample_5.wav (Duration: 15.60s)
Done!


In [20]:
target_dir = "../resources/music"

# Create a folder for your samples

os.makedirs(target_dir, exist_ok=True)

# 1. Load in streaming mode (no full download)
# We'll use 'train.360' to ensure we find longer clips quickly
ds = load_dataset("sanchit-gandhi/gtzan", split="train", streaming=True)

JAZZ_ID = 5

jazz_ds = ds.filter(lambda example: example["genre"] == JAZZ_ID)

found_count = 0
target_count = 5
min_duration_sec = 30
target_duration_sec = 15

print(f"Searching for samples >= {min_duration_sec}s...")

for sample in jazz_ds:
    audio = sample["audio"]
    file_like_object = io.BytesIO(audio["bytes"])
    array, sampling_rate = sf.read(file_like_object)
    
    # Calculate duration: total samples / samples per second
    duration = len(array) / sampling_rate
    
    if duration >= min_duration_sec:
        found_count += 1
        file_name = os.path.join(target_dir,f"sample_{found_count}.wav")

        # Truncate to center target_duration_sec of audio
        if duration > target_duration_sec:
            start_sample = int(((duration - target_duration_sec) / 2) * sampling_rate)
            end_sample = start_sample + int(target_duration_sec * sampling_rate)
            array = array[start_sample:end_sample]
        
        # Save as WAV
        sf.write(file_name, array, sampling_rate)
        
        print(f"Saved {file_name} (Duration: {duration:.2f}s)")
    else:
        print(f"Skipping - (Duration: {duration:.2f}s)")
        
    if found_count >= target_count:
        break

print("Done!")

Searching for samples >= 30s...
Saved ../resources/music/sample_1.wav (Duration: 30.01s)
Saved ../resources/music/sample_2.wav (Duration: 30.01s)
Saved ../resources/music/sample_3.wav (Duration: 30.01s)
Saved ../resources/music/sample_4.wav (Duration: 30.01s)
Saved ../resources/music/sample_5.wav (Duration: 30.01s)
Done!


In [3]:
from collections import Counter
stats = Counter()

# Configuration
target_dir = "../resources/sfx"
os.makedirs(target_dir, exist_ok=True)

# The categories you requested
target_categories = [
    "thunderstorm", "fireworks", "clapping", "car_horn", 
    "glass_breaking", "sea_waves", "crickets", "engine"
]

# 1. Load in streaming mode
ds = load_dataset("ashraq/esc50", split="train", streaming=True)

# 2. Filter for only your requested categories
filtered_ds = ds.filter(lambda x: x["category"] in target_categories)

found_count = 0
target_total = 15

print(f"Searching for up to {target_total} samples from categories: {target_categories}...")

for sample in filtered_ds:
    category = sample["category"]
    audio = sample["audio"]
    file_like_object = io.BytesIO(audio["bytes"])
    array, sampling_rate = sf.read(file_like_object)
    
    if stats[category] < 2:  # Limit each category to 2 samples
        stats[category] += 1

        # Generate a descriptive filename
        found_count += 1
        file_name = os.path.join(target_dir, f"{category}_{found_count}.wav")
        
        # Save the file
        sf.write(file_name, array, sampling_rate)
        print(f"[{found_count}/{target_total}] Saved {file_name}")
        
        if found_count >= target_total:
            break

print("\nDownload complete!")

Repo card metadata block was not found. Setting CardData to empty.


Searching for up to 15 samples from categories: ['thunderstorm', 'fireworks', 'clapping', 'car_horn', 'glass_breaking', 'sea_waves', 'crickets', 'engine']...
[1/15] Saved ../resources/sfx/thunderstorm_1.wav
[2/15] Saved ../resources/sfx/thunderstorm_2.wav
[3/15] Saved ../resources/sfx/clapping_3.wav
[4/15] Saved ../resources/sfx/clapping_4.wav
[5/15] Saved ../resources/sfx/fireworks_5.wav
[6/15] Saved ../resources/sfx/fireworks_6.wav
[7/15] Saved ../resources/sfx/car_horn_7.wav
[8/15] Saved ../resources/sfx/engine_8.wav
[9/15] Saved ../resources/sfx/engine_9.wav
[10/15] Saved ../resources/sfx/car_horn_10.wav
[11/15] Saved ../resources/sfx/glass_breaking_11.wav
[12/15] Saved ../resources/sfx/sea_waves_12.wav
[13/15] Saved ../resources/sfx/sea_waves_13.wav
[14/15] Saved ../resources/sfx/crickets_14.wav
[15/15] Saved ../resources/sfx/crickets_15.wav

Download complete!


In [25]:
audio.keys()

dict_keys(['bytes', 'path'])